# Week 6 — Image Segmentation for Object-Based Image Analysis (GEOBIA)
**Geoprocesamiento · Maestría en Geomática · UNAL**

This notebook builds a complete GEOBIA pipeline that reuses every cloud-native format from Week 5:

```
STAC (search) ──► COG (stream only the AOI) ──► segmentation ──► image objects + attributes ──► GeoParquet
```

**Base example:** flower greenhouses in the Sabana de Bogotá (Madrid – Funza – Mosquera), Sentinel-2 L2A from Element 84 Earth Search.

> ✏️ **Your task:** edit only the cell marked `👉 CONFIGURE YOUR AOI`, choose an object type that makes sense in your area, and justify your segmentation parameters with the evaluation in Stage 4.

In [ ]:
# Run once in your environment (JupyterLab) if the libraries are missing
# !pip install pystac-client rioxarray scikit-image scikit-learn geopandas pyarrow folium mapclassify matplotlib

In [ ]:
import os
import time
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray
from pystac_client import Client
from rasterio.enums import Resampling
from rasterio.features import shapes
from shapely.geometry import box, shape
from skimage.segmentation import mark_boundaries
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

## 👉 CONFIGURE YOUR AOI
Keep the AOI around **10 × 10 km** (≈ 1,000 × 1,000 pixels at 10 m) so segmentation runs in seconds and fits in RAM.
Use [geojson.io](https://geojson.io) or [bboxfinder.com](http://bboxfinder.com) to get your bounding box in `[min_lon, min_lat, max_lon, max_lat]`.

In [ ]:
AOI_NAME   = "sabana_greenhouses"
BBOX       = [-74.30, 4.70, -74.20, 4.80]      # [min_lon, min_lat, max_lon, max_lat] (EPSG:4326)
DATE_RANGE = "2025-12-01/2026-03-31"           # dry season = fewer clouds in the Andes
MAX_CLOUD  = 20                                 # % scene cloud cover
OBJECT_OF_INTEREST = "greenhouses"              # what are YOU trying to delineate?

OUTPUT_DIR = f"outputs_{AOI_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# =============================================================================
# STAGE 1: STAC SEARCH (Week 5 recap)
# =============================================================================
# WHAT: Query the Earth Search STAC API for Sentinel-2 L2A scenes over the AOI.
# HOW:  Filter by bbox, date range and eo:cloud_cover; keep scenes whose footprint
#       fully contains the AOI and pick the least cloudy one.
# RESULT: One STAC Item whose Assets are COGs (one file per band).
# =============================================================================
catalog = Client.open("https://earth-search.aws.element84.com/v1")
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=BBOX,
    datetime=DATE_RANGE,
    query={"eo:cloud_cover": {"lt": MAX_CLOUD}},
    max_items=100,
)
items = list(search.items())
aoi_geom = box(*BBOX)
items = [it for it in items if shape(it.geometry).contains(aoi_geom)]
items = sorted(items, key=lambda it: it.properties["eo:cloud_cover"])
print(f"✓ {len(items)} scenes fully cover the AOI")
if not items:
    raise RuntimeError("No scenes found: widen DATE_RANGE, raise MAX_CLOUD or shrink BBOX.")

item = items[0]
print(f"✓ Selected: {item.id} | {item.datetime:%Y-%m-%d} | cloud {item.properties['eo:cloud_cover']:.1f}%")
print(f"✓ Assets used: {[k for k in ['blue', 'green', 'red', 'nir', 'swir16', 'scl'] if k in item.assets]}")

In [ ]:
# =============================================================================
# STAGE 2: STREAM THE AOI FROM THE COGs (Week 5 recap) + CLOUD MASK + NDVI
# =============================================================================
# WHAT: Read only the tiles that intersect the AOI from each band's COG.
# HOW:  rioxarray.open_rasterio() reads the header; clip_box() computes the window
#       (the AOI is given in EPSG:4326 and transformed to the scene's UTM CRS);
#       .load() fetches just those bytes. 20 m bands are resampled to the 10 m grid.
# RESULT: Reflectance arrays on a common 10 m grid + a valid-pixel mask.
# =============================================================================
def read_asset(item, key, scale_reflectance=True):
    asset = item.assets[key]
    da = rioxarray.open_rasterio(asset.href, masked=True).squeeze("band", drop=True)
    da = da.rio.clip_box(*BBOX, crs="EPSG:4326").load()
    if scale_reflectance:
        rb = (asset.extra_fields.get("raster:bands") or [{}])[0]
        da = da * rb.get("scale", 0.0001) + rb.get("offset", 0.0)
    return da

t0 = time.time()
red = read_asset(item, "red")                               # 10 m reference grid
bands = {
    "blue":   read_asset(item, "blue"),
    "green":  read_asset(item, "green"),
    "red":    red,
    "nir":    read_asset(item, "nir"),
    "swir16": read_asset(item, "swir16").rio.reproject_match(red, resampling=Resampling.bilinear),
}
scl = read_asset(item, "scl", scale_reflectance=False).rio.reproject_match(red, resampling=Resampling.nearest)
print(f"✓ Streamed {len(bands) + 1} bands in {time.time() - t0:.1f} s | grid {red.shape} | CRS {red.rio.crs}")

# Scene Classification Layer: drop no-data, saturated, cloud shadow, clouds and cirrus
BAD_SCL = [0, 1, 3, 8, 9, 10]
arrays = {k: v.values.astype("float32") for k, v in bands.items()}
valid = ~np.isin(scl.values, BAD_SCL) & np.all([np.isfinite(a) for a in arrays.values()], axis=0)
arrays["ndvi"] = (arrays["nir"] - arrays["red"]) / (arrays["nir"] + arrays["red"] + 1e-9)
print(f"✓ Valid pixels: {valid.mean():.1%}")

TRANSFORM, CRS = red.rio.transform(), red.rio.crs
PIXEL_AREA = abs(TRANSFORM.a * TRANSFORM.e)                  # 100 m² for 10 m pixels

In [ ]:
def stretch(a, valid):
    lo, hi = np.nanpercentile(a[valid], [2, 98])
    return np.clip((a - lo) / (hi - lo), 0, 1)

rgb = np.dstack([stretch(arrays[b], valid) for b in ("red", "green", "blue")])
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(rgb); ax[0].set_title("True colour (B4-B3-B2)")
im = ax[1].imshow(np.where(valid, arrays["ndvi"], np.nan), cmap="RdYlGn", vmin=-0.2, vmax=0.9)
ax[1].set_title("NDVI"); plt.colorbar(im, ax=ax[1], shrink=0.7)
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## Segmentation helpers
Pure NumPy / scikit-image / pandas functions used in the next stages. Read the docstrings — you don't need to edit this cell.

In [ ]:
# =============================================================================
# SEGMENTATION HELPERS (pure NumPy / scikit-image / pandas)
# =============================================================================
import numpy as np
import pandas as pd
from skimage.segmentation import slic, felzenszwalb
from skimage.measure import regionprops_table


def normalize_stack(stack, valid, p_low=2, p_high=98):
    """Stretch each band to [0, 1] using percentiles of valid pixels.

    stack : np.ndarray (C, H, W)  -> reflectance bands
    valid : np.ndarray (H, W) bool -> True where the pixel is usable
    returns np.ndarray (H, W, C) float32, the layout scikit-image expects
    """
    out = np.zeros(stack.shape[1:] + (stack.shape[0],), dtype="float32")
    for i, band in enumerate(stack):
        lo, hi = np.nanpercentile(band[valid], [p_low, p_high])
        out[..., i] = np.clip((band - lo) / (hi - lo + 1e-12), 0, 1)
    out[~valid] = 0
    return out


def run_segmentation(img, valid, method, **params):
    """Run one segmentation algorithm and return an int32 label image.

    Label 0 = no data (masked); labels 1..N = segments (image objects).
    """
    if method == "slic":
        labels = slic(img, channel_axis=-1, start_label=1, mask=valid, **params)
    elif method == "felzenszwalb":
        labels = felzenszwalb(img, channel_axis=-1, **params) + 1
        labels[~valid] = 0
    else:
        raise ValueError(f"Unknown method: {method}")
    return labels.astype("int32")


def segment_adjacency(labels):
    """Return unique pairs (i, j) of segments that share a pixel edge (rook)."""
    pairs = []
    for a, b in [(labels[:, :-1], labels[:, 1:]), (labels[:-1, :], labels[1:, :])]:
        m = (a != b) & (a > 0) & (b > 0)
        pairs.append(np.stack([a[m], b[m]], axis=1))
    pairs = np.concatenate(pairs)
    return np.unique(np.sort(pairs, axis=1), axis=0)


def johnson_xie(labels, band):
    """Unsupervised segmentation quality (Johnson & Xie, 2011).

    - wVar   : area-weighted variance INSIDE segments  (low = homogeneous objects)
    - MoranI : spatial autocorrelation BETWEEN neighbour segment means
               (low = neighbours are different from each other)
    Under-segmentation -> high wVar ; over-segmentation -> high Moran's I.
    """
    flat, v = labels.ravel(), band.ravel()
    m = (flat > 0) & np.isfinite(v)
    g = pd.DataFrame({"id": flat[m], "v": v[m]}).groupby("id")["v"]
    area, var, mean = g.size(), g.var(ddof=0), g.mean()
    wvar = float((area * var).sum() / area.sum())

    pairs = segment_adjacency(labels)
    keep = np.isin(pairs[:, 0], mean.index) & np.isin(pairs[:, 1], mean.index)
    pairs = pairs[keep]
    dev = mean - mean.mean()
    di, dj = dev.loc[pairs[:, 0]].to_numpy(), dev.loc[pairs[:, 1]].to_numpy()
    n, w_sum = len(mean), 2 * len(pairs)          # symmetric binary weights
    moran = float((n / w_sum) * (2 * np.sum(di * dj)) / np.sum(dev.to_numpy() ** 2))
    return wvar, moran


def evaluate_parameters(img, valid, band, method, param_grid):
    """Segment with every parameter set and score it with Johnson & Xie."""
    rows = []
    for params in param_grid:
        labels = run_segmentation(img, valid, method, **params)
        wvar, moran = johnson_xie(labels, band)
        rows.append({"method": method, **params,
                     "n_segments_out": int(labels.max()),
                     "wVar": wvar, "MoranI": moran})
    df = pd.DataFrame(rows)
    norm = lambda s: (s - s.min()) / (s.max() - s.min() + 1e-12)
    df["GS"] = norm(df["wVar"]) + norm(df["MoranI"])   # Global Score: lower = better
    return df.sort_values("GS").reset_index(drop=True)


def object_features(labels, bands, pixel_area_m2):
    """Spectral + geometric attributes per segment (one row = one object)."""
    flat = labels.ravel()
    m = flat > 0
    df = pd.DataFrame({"segment_id": flat[m]})
    for name, arr in bands.items():
        df[name] = arr.ravel()[m]
    g = df.groupby("segment_id")
    feats = g.mean().add_suffix("_mean").join(g.std().add_suffix("_std"))
    feats["n_pixels"] = g.size()
    feats["area_m2"] = feats["n_pixels"] * pixel_area_m2

    props = pd.DataFrame(regionprops_table(
        labels, properties=("label", "eccentricity", "solidity", "extent")
    )).rename(columns={"label": "segment_id"}).set_index("segment_id")
    return feats.join(props).reset_index()

In [ ]:
# =============================================================================
# STAGE 3: SEGMENTATION — COMPARE ALGORITHMS AND PARAMETERS
# =============================================================================
# WHAT: Group neighbouring, spectrally similar pixels into image objects.
# HOW:  Input = 5 stretched bands (B, G, R, NIR, SWIR1). Two algorithms:
#       • SLIC (superpixels): n_segments ≈ how many objects; compactness = shape
#         regularity vs spectral fidelity (low = follows edges, high = square-ish).
#       • Felzenszwalb (graph-based): scale = larger → bigger objects;
#         min_size = smallest allowed object in pixels.
# RESULT: Label images (0 = masked, 1..N = segments) overlaid on true colour.
# =============================================================================
stack = np.stack([arrays[b] for b in ("blue", "green", "red", "nir", "swir16")])
img = normalize_stack(stack, valid)

examples = [
    ("slic",         {"n_segments": 500,  "compactness": 0.5}),
    ("slic",         {"n_segments": 3000, "compactness": 0.5}),
    ("felzenszwalb", {"scale": 50,  "sigma": 0.5, "min_size": 20}),
    ("felzenszwalb", {"scale": 800, "sigma": 0.5, "min_size": 20}),
]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, (method, params) in zip(axes.ravel(), examples):
    t0 = time.time()
    labels = run_segmentation(img, valid, method, **params)
    ax.imshow(mark_boundaries(rgb, labels, color=(1, 1, 0)))
    ax.set_title(f"{method} {params}\n{labels.max()} segments · {time.time() - t0:.1f} s", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

### 🔎 Look before you measure
Zoom into the figures: where do you see **over-segmentation** (one greenhouse split into many pieces) and **under-segmentation** (a greenhouse merged with a road or pasture)? Write it down — you'll compare your visual judgement with the metric below.

In [ ]:
# =============================================================================
# STAGE 4: EVALUATE SEGMENTATION QUALITY (Johnson & Xie, 2011)
# =============================================================================
# WHAT: Choose parameters with an unsupervised, reproducible criterion.
# HOW:  For each parameter set compute
#       • wVar   — area-weighted variance inside segments (↑ = under-segmentation)
#       • MoranI — autocorrelation between neighbouring segments (↑ = over-segmentation)
#       Both are min-max normalised and summed → Global Score (GS). Lower = better.
# RESULT: A ranked table + the best parameter set for each algorithm.
# NOTE:  The metric is computed on NDVI; try another band relevant to YOUR objects.
# =============================================================================
slic_grid = [{"n_segments": n, "compactness": c} for n in (500, 1500, 3000) for c in (0.1, 0.5, 2.0)]
felz_grid = [{"scale": s, "sigma": 0.5, "min_size": 20} for s in (50, 200, 800)]

eval_slic = evaluate_parameters(img, valid, arrays["ndvi"], "slic", slic_grid)
eval_felz = evaluate_parameters(img, valid, arrays["ndvi"], "felzenszwalb", felz_grid)
display(eval_slic.round(4))
display(eval_felz.round(4))

fig, ax = plt.subplots(figsize=(6, 5))
for df, mk in [(eval_slic, "o"), (eval_felz, "s")]:
    ax.scatter(df["MoranI"], df["wVar"], marker=mk, label=df["method"].iloc[0])
ax.set_xlabel("Moran's I between segments (over-segmentation →)")
ax.set_ylabel("Weighted variance within segments (under-segmentation →)")
ax.legend(); ax.set_title("Parameter trade-off (bottom-left is best)")
plt.tight_layout(); plt.show()

In [ ]:
# Pick the winner (you may override it if your visual judgement disagrees — justify it!)
best = eval_slic.iloc[0]
BEST_METHOD = "slic"
BEST_PARAMS = {"n_segments": int(best["n_segments"]), "compactness": float(best["compactness"])}
labels = run_segmentation(img, valid, BEST_METHOD, **BEST_PARAMS)
print(f"✓ Using {BEST_METHOD} {BEST_PARAMS} → {labels.max()} segments")

In [ ]:
# =============================================================================
# STAGE 5: OBJECT ATTRIBUTES — FROM PIXELS TO FEATURES
# =============================================================================
# WHAT: Describe every object with spectral and geometric attributes.
# HOW:  Group pixels by segment id → mean/std of each band and NDVI, area,
#       eccentricity (elongation), solidity (convexity), extent (box filling).
# RESULT: A table with ONE ROW PER OBJECT — this is what OBIA classifies.
# =============================================================================
feats = object_features(labels, arrays, PIXEL_AREA)
print(f"✓ {len(feats)} objects × {feats.shape[1]} attributes")
feats.describe().T.round(3)

In [ ]:
# =============================================================================
# STAGE 6: RASTER → VECTOR (image objects as polygons)
# =============================================================================
# WHAT: Convert the label image into polygons and attach the attribute table.
# HOW:  rasterio.features.shapes() traces the boundary of each label using the
#       COG's affine transform; dissolve() merges parts of the same segment.
# RESULT: A GeoDataFrame in the scene's UTM CRS (metres).
# =============================================================================
t0 = time.time()
polys = [(shape(geom), int(val)) for geom, val in shapes(labels, mask=labels > 0, transform=TRANSFORM)]
gdf = gpd.GeoDataFrame(
    {"segment_id": [v for _, v in polys]},
    geometry=[g for g, _ in polys],
    crs=CRS,
).dissolve(by="segment_id").reset_index()

gdf["perimeter_m"] = gdf.geometry.length
gdf["polsby_popper"] = 4 * np.pi * gdf.geometry.area / gdf["perimeter_m"] ** 2   # 1 = circle
gdf = gdf.merge(feats, on="segment_id")
gdf["scene_id"], gdf["acq_date"] = item.id, item.datetime.date().isoformat()
print(f"✓ {len(gdf)} polygons in {time.time() - t0:.1f} s | CRS {gdf.crs}")
gdf.head()

In [ ]:
# =============================================================================
# STAGE 7: SAVE AS GEOPARQUET (Week 5 recap) AND COMPARE WITH GEOJSON
# =============================================================================
# WHAT: Persist the objects in a cloud-native, columnar vector format.
# HOW:  to_parquet() keeps the UTM CRS in the 'geo' footer metadata;
#       GeoJSON must be WGS 84 (RFC 7946), so we reproject before writing it.
# RESULT: File-size comparison + a column-projection read.
# =============================================================================
pq_path = os.path.join(OUTPUT_DIR, f"{AOI_NAME}_objects.parquet")
gj_path = os.path.join(OUTPUT_DIR, f"{AOI_NAME}_objects.geojson")
gdf.to_parquet(pq_path)
gdf.to_crs(4326).to_file(gj_path, driver="GeoJSON")

size_pq, size_gj = os.path.getsize(pq_path) / 1e6, os.path.getsize(gj_path) / 1e6
print(f"GeoParquet: {size_pq:6.2f} MB")
print(f"GeoJSON   : {size_gj:6.2f} MB  → GeoParquet is {size_gj / size_pq:.1f}× smaller")

# Column projection: read only what you need
subset = gpd.read_parquet(pq_path, columns=["segment_id", "ndvi_mean", "area_m2", "geometry"])
print(f"✓ Read back {subset.shape[1]} of {gdf.shape[1]} columns")

In [ ]:
# =============================================================================
# STAGE 8: A FIRST LOOK AT OBJECT CLASSIFICATION (unsupervised)
# =============================================================================
# WHAT: Group objects with similar attributes — a preview of next week.
# HOW:  Standardise spectral + shape attributes and run K-Means.
# RESULT: An interactive map; click objects to inspect their attributes.
# =============================================================================
FEATURES = ["ndvi_mean", "ndvi_std", "red_mean", "nir_mean", "swir16_mean",
            "blue_mean", "polsby_popper", "solidity", "eccentricity"]
X = StandardScaler().fit_transform(gdf[FEATURES].fillna(0))
gdf["cluster"] = KMeans(n_clusters=6, n_init=10, random_state=42).fit_predict(X).astype(str)

display(gdf.groupby("cluster")[["ndvi_mean", "blue_mean", "area_m2", "polsby_popper"]].mean().round(3))

gdf.to_crs(4326).explore(
    column="cluster",
    categorical=True,
    cmap="tab10",
    tooltip=["segment_id", "cluster", "ndvi_mean", "area_m2", "polsby_popper"],
    style_kwds=dict(weight=0.3, fillOpacity=0.5),
    tiles="Esri.WorldImagery",
)

## ✅ Your turn — deliverables
1. **AOI & object:** your bounding box, date range and the object type you want to delineate (and why it suits OBIA).
2. **Segmentation:** at least **2 algorithms × 3 parameter values**, shown as boundary overlays.
3. **Evaluation:** the Johnson & Xie table/plot and your final choice — does the metric agree with your visual judgement? If not, which do you trust and why?
4. **Objects:** your `*_objects.parquet` file and the interactive map.
5. **Reflection (½ page):** which cluster(s) correspond to your objects of interest? Which attributes separate them best? What would you need to turn this into a supervised classification?

**References:** Blaschke (2010) *ISPRS J. Photogramm.* 65(1) · Johnson & Xie (2011) *ISPRS J. Photogramm.* 66(4) · Achanta et al. (2012) SLIC, *IEEE TPAMI* · Felzenszwalb & Huttenlocher (2004) *IJCV*.